# EDA COVID-19: evoluci?n de la pandemia en Estados Unidos

## 1. Contexto y objetivo

Este proyecto desarrolla un an?lisis exploratorio del conjunto hist?rico de datos de COVID-19 publicado por The COVID Tracking Project.

**Periodo analizado:** enero de 2020 ? marzo de 2021.

**Objetivo:** estudiar la evoluci?n temporal de la pandemia, explorar diferencias entre jurisdicciones y evaluar la calidad y las limitaciones de los datos.

### Preguntas de investigaci?n

1. ?C?mo evolucionaron los nuevos casos y fallecimientos a lo largo del periodo?
2. ?Qu? patrones temporales se observan en los registros?
3. ?Qu? jurisdicciones presentan los mayores valores acumulados reportados?
4. ?Qu? variables tienen mayor ausencia de informaci?n?
5. ?Qu? limitaciones debemos considerar antes de interpretar los resultados?

> Los resultados son descriptivos. No permiten establecer causalidad ni comparar directamente el riesgo entre jurisdicciones sin considerar sus poblaciones y otros factores.


## 2. Librer?as y configuraci?n

Importamos las herramientas necesarias para manipular, analizar y visualizar los datos.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

sns.set_theme(style="whitegrid", context="notebook")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "covid_us_states_daily.csv"

print(f"Directorio del proyecto: {PROJECT_ROOT}")
print(f"Dataset: {DATA_PATH}")
print(f"Existe el archivo: {DATA_PATH.exists()}")

## 3. Carga y comprensi?n inicial

Leemos el CSV original. Conservamos una copia sin transformar y creamos otra para el an?lisis.

La variable `date` viene codificada como entero con formato `YYYYMMDD`. La convertiremos a fecha real para poder analizar correctamente la dimensi?n temporal.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"No se encontr? el dataset en {DATA_PATH}. "
        "Comprueba la ubicaci?n del archivo."
    )

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()

df["date"] = pd.to_datetime(
    df["date"].astype("string"),
    format="%Y%m%d",
    errors="coerce"
)

df = df.sort_values(["state", "date"]).reset_index(drop=True)

print(f"Registros: {len(df):,}")
print(f"Variables: {df.shape[1]}")
print(f"Jurisdicciones: {df['state'].nunique()}")
print(f"Fecha inicial: {df['date'].min().date()}")
print(f"Fecha final: {df['date'].max().date()}")

display(df.head())

## 4. Calidad y estructura de los datos

Primero comprobamos los tipos, los registros duplicados y la cobertura temporal.

No eliminamos autom?ticamente duplicados, nulos ni valores negativos: antes de tomar decisiones hay que comprender su origen y significado.


In [ ]:
print("TIPOS DE DATOS")
display(df.dtypes.rename("tipo").to_frame())

print("DUPLICADOS EXACTOS")
print(df.duplicated().sum())

print("DUPLICADOS POR JURISDICCI?N Y FECHA")
print(df.duplicated(subset=["state", "date"]).sum())

print("COBERTURA POR JURISDICCI?N")
coverage = (
    df.groupby("state")
      .agg(
          first_date=("date", "min"),
          last_date=("date", "max"),
          records=("date", "count"),
          unique_dates=("date", "nunique")
      )
      .sort_values("records")
)

display(coverage.head(10))
display(coverage.tail(10))

## 5. An?lisis de valores ausentes

La ausencia de informaci?n puede variar considerablemente entre variables. Calculamos tanto el n?mero como el porcentaje de valores ausentes.

Un valor nulo no equivale necesariamente a cero: puede significar que una jurisdicci?n no report? ese dato o que la variable no estaba disponible.


In [ ]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100
}).sort_values("missing_pct", ascending=False)

display(missing.head(20))

missing_to_plot = missing[missing["missing_count"] > 0].head(15)

plt.figure(figsize=(11, 7))
sns.barplot(
    data=missing_to_plot.reset_index(names="variable"),
    x="missing_pct",
    y="variable"
)
plt.title("Variables con mayor porcentaje de valores ausentes")
plt.xlabel("Valores ausentes (%)")
plt.ylabel("Variable")
plt.tight_layout()
plt.show()

## 6. Evoluci?n temporal de la pandemia

Comenzamos con tres indicadores diarios:

- `positiveIncrease`: nuevos casos positivos reportados.
- `deathIncrease`: nuevos fallecimientos reportados.
- `totalTestResultsIncrease`: incremento diario de resultados de pruebas.

Sumamos los registros por fecha para obtener una serie agregada de las jurisdicciones incluidas. Calculamos tambi?n una media m?vil de siete d?as para facilitar la observaci?n de tendencias.

**Precauci?n:** las cifras agregadas dependen de la cobertura y de las pr?cticas de notificaci?n. Los valores negativos pueden reflejar revisiones retrospectivas.


In [ ]:
daily_variables = [
    "positiveIncrease",
    "deathIncrease",
    "totalTestResultsIncrease"
]

daily = (
    df.groupby("date")[daily_variables]
      .sum(min_count=1)
      .sort_index()
)

daily_rolling = daily.rolling(window=7, min_periods=1).mean()

fig, axes = plt.subplots(3, 1, figsize=(14, 13), sharex=True)

titles = [
    "Nuevos casos positivos reportados",
    "Nuevos fallecimientos reportados",
    "Incremento de resultados de pruebas"
]

for ax, column, title in zip(axes, daily_variables, titles):
    ax.plot(daily.index, daily[column], alpha=0.30, label="Valor diario")
    ax.plot(
        daily_rolling.index,
        daily_rolling[column],
        linewidth=2,
        label="Media m?vil de 7 d?as"
    )
    ax.set_title(title)
    ax.set_ylabel("Registros")
    ax.legend()

axes[-1].set_xlabel("Fecha")
plt.tight_layout()
plt.show()

## 7. Comparaci?n exploratoria entre jurisdicciones

Para evitar comparar fechas distintas, seleccionamos el ?ltimo valor disponible de `positive` para cada jurisdicci?n.

Esto permite una primera comparaci?n descriptiva, pero las fechas de actualizaci?n pueden diferir. Por tanto, no debemos interpretar el ranking como una comparaci?n perfectamente simult?nea ni como una medida de riesgo poblacional.


In [ ]:
latest_positive = (
    df.dropna(subset=["positive"])
      .sort_values("date")
      .groupby("state", as_index=False)
      .tail(1)
      .sort_values("positive", ascending=False)
)

top_10 = latest_positive.head(10).copy()

display(
    top_10[["state", "date", "positive"]]
    .reset_index(drop=True)
)

plt.figure(figsize=(11, 6))
sns.barplot(
    data=top_10,
    x="positive",
    y="state"
)
plt.title("10 jurisdicciones con mayor ?ltimo valor acumulado de positivos")
plt.xlabel("Casos positivos acumulados reportados")
plt.ylabel("Jurisdicci?n")
plt.tight_layout()
plt.show()

## 8. Primeras conclusiones y limitaciones

Completar esta secci?n despu?s de ejecutar y revisar las visualizaciones.

### Limitaciones que debemos tener presentes

- El conjunto contiene 56 jurisdicciones, no exclusivamente estados.
- La cobertura temporal y la disponibilidad de variables no son uniformes.
- Los valores ausentes no deben sustituirse autom?ticamente por cero.
- Los incrementos diarios negativos pueden reflejar correcciones de los reportes.
- Los valores acumulados absolutos no permiten comparar el riesgo sin considerar la poblaci?n.
- Los ?ltimos valores disponibles pueden corresponder a fechas diferentes.
- La media m?vil suaviza fluctuaciones, pero no corrige problemas de calidad ni de notificaci?n.

### Pr?ximas l?neas de an?lisis

1. Investigar distribuciones y asimetr?as de las variables principales.
2. Comparar tendencias entre jurisdicciones seleccionadas.
3. Analizar la relaci?n entre casos, fallecimientos y pruebas.
4. Evaluar indicadores normalizados por poblaci?n si incorporamos una fuente demogr?fica fiable.
5. Documentar los hallazgos con sus limitaciones y decisiones metodol?gicas.
